# Team-week efficiency and explosiveness

## Pipeline overview

**Purpose:** Build team-game-week metrics for the Streamlit team dashboard. Regular-season and postseason games are included when PBP contains an `END GAME` record. Future or incomplete games are reported but not published.

| Role | File | Current columns | Grain |
| --- | --- | ---: | --- |
| Primary input | `data/silver/pbp/silver_pbp_{season}.parquet` | 372 | `game_id + play_id` |
| Context and validation | `data/silver/schedules/silver_schedules_{season}.parquet` | 46 | `game_id` |
| Output | `data/gold/team_week_metrics/gold_team_week_metrics_{season}.parquet` | 17 | `season + week + season_type + game_id + team` |

**Seasons:** 2023–2026. The current season is in progress, so no row totals are hard-coded.

**Executed snapshot (2026-10-17):** 935 completed games produced 1,870 team rows across four 17-column files. The 2026 partition has 80 games and includes `2026_06_SEA_DEN`, whose schedule score was still null. All checks below passed.

### Transformations and definitions

1. Select completed PBP games with an `END GAME` marker; reconcile their schedule context. Schedule scores are a freshness diagnostic, not a completion gate.
2. Use plays with offense and defense teams, `play_type` of `pass` or `run`, and `two_point_attempt == 0`. The `no_play` and other play types do not enter the metric population.
3. Mark explosive passes at 16+ yards and explosive runs at 12+ yards. Preserve source `epa` and `success` values.
4. Group by `posteam` for offense and `defteam` for defense. Defensive EPA *allowed* has the same sign as the opponent offense's EPA; it is not negated.
5. Retain additive numerators and denominators so season-level rates can be recomputed as sums divided by sums.

| Rate | Numerator / denominator |
| --- | --- |
| `offensive_epa_per_play` | `offensive_epa_sum / offensive_plays` |
| `defensive_epa_per_play_allowed` | `defensive_epa_allowed_sum / defensive_plays_faced` |
| `offensive_success_rate` | `offensive_successful_plays / offensive_plays` |
| `offensive_explosive_play_rate` | `offensive_explosive_plays / offensive_plays` |

### Validation and quality checks

- Required files, columns, unique Silver grains, and exact PBP-to-schedule game context.
- Completed-game diagnostic, including missing schedule scores without incorrectly dropping completed games.
- Complete eligible EPA, success, and yard values; binary success agrees with positive EPA.
- Exactly two team rows per included game, unique Gold grain, positive play denominators, valid opponents, and bounded rates.
- Offensive totals reconcile to eligible plays; each offense's counts and signed EPA match the opposing defense's allowed values.
- Each season partition survives a Parquet round trip with its 17-column schema and values intact.


## 1. Set paths and the output contract


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

SEASONS = [2023, 2024, 2025, 2026]
PLAY_KEY = ["game_id", "play_id"]
GOLD_KEY = ["season", "week", "season_type", "game_id", "team"]
GROUP_KEYS = ["season", "week", "season_type", "game_id"]
PBP_COLUMNS = [
    "game_id", "play_id", "season", "week", "season_type",
    "posteam", "defteam", "play_type", "two_point_attempt",
    "epa", "success", "yards_gained", "desc",
]
SCHEDULE_COLUMNS = [
    "game_id", "season", "week", "game_type", "gameday",
    "home_team", "away_team", "home_score", "away_score",
]
OUTPUT_COLUMNS = [
    "season", "week", "season_type", "game_id", "game_date",
    "team", "opponent_team",
    "offensive_plays", "offensive_epa_sum",
    "offensive_successful_plays", "offensive_explosive_plays",
    "defensive_plays_faced", "defensive_epa_allowed_sum",
    "offensive_epa_per_play", "defensive_epa_per_play_allowed",
    "offensive_success_rate", "offensive_explosive_play_rate",
]

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "ROADMAP.md").exists():
    PROJECT_ROOT = next(
        parent for parent in PROJECT_ROOT.parents
        if (parent / "ROADMAP.md").exists()
    )
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
GOLD_DIR = PROJECT_ROOT / "data" / "gold" / "team_week_metrics"
assert len(OUTPUT_COLUMNS) == 17
print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/dtwice/Development/dev_sports/nfl/fantasy-football/nfl-dream-lab-app


## 2. Load and validate Silver inputs

Read only the fields needed for this product. Input column counts are checked before projection.


In [2]:
play_partitions = []
schedule_partitions = []
input_summary = []

for season in SEASONS:
    pbp_path = SILVER_DIR / "pbp" / f"silver_pbp_{season}.parquet"
    schedule_path = SILVER_DIR / "schedules" / f"silver_schedules_{season}.parquet"
    assert pbp_path.exists(), f"Missing PBP: {pbp_path}"
    assert schedule_path.exists(), f"Missing schedule: {schedule_path}"
    pbp_schema = pq.read_schema(pbp_path)
    schedule_schema = pq.read_schema(schedule_path)
    assert len(pbp_schema) == 372 and set(PBP_COLUMNS).issubset(pbp_schema.names)
    assert len(schedule_schema) == 46 and set(SCHEDULE_COLUMNS).issubset(schedule_schema.names)

    season_plays = pd.read_parquet(pbp_path, columns=PBP_COLUMNS)
    season_schedules = pd.read_parquet(schedule_path, columns=SCHEDULE_COLUMNS)
    assert season_plays["season"].eq(season).all()
    assert season_schedules["season"].eq(season).all()
    play_partitions.append(season_plays)
    schedule_partitions.append(season_schedules)
    input_summary.append({
        "season": season, "pbp_rows": len(season_plays),
        "schedule_rows": len(season_schedules),
    })

plays = pd.concat(play_partitions, ignore_index=True)
schedules = pd.concat(schedule_partitions, ignore_index=True)
assert plays[PLAY_KEY].notna().all().all()
assert not plays.duplicated(PLAY_KEY).any()
assert schedules["game_id"].notna().all()
assert not schedules.duplicated("game_id").any()
pd.DataFrame(input_summary)


,season,pbp_rows,schedule_rows
0,2023,49665,285
1,2024,49492,285
2,2025,48771,285
3,2026,14031,272


## 3. Reconcile games and identify completed PBP

A schedule score can arrive after PBP. `END GAME` is the publication gate for this prototype; absent markers are listed for investigation and withheld.


In [3]:
pbp_game_context = plays[["game_id", "season", "week", "season_type"]].drop_duplicates()
assert not pbp_game_context.duplicated("game_id").any()
game_context = pbp_game_context.merge(
    schedules, on="game_id", how="left", validate="one_to_one",
    indicator=True, suffixes=("_pbp", "_schedule"),
)
assert game_context["_merge"].eq("both").all()
assert game_context["season_pbp"].eq(game_context["season_schedule"]).all()
assert game_context["week_pbp"].eq(game_context["week_schedule"]).all()
schedule_season_type = game_context["game_type"].replace(
    {game_type: "POST" for game_type in ["WC", "DIV", "CON", "SB"]}
)
assert game_context["season_type"].eq(schedule_season_type).all()
assert game_context["home_team"].ne(game_context["away_team"]).all()

completed_game_ids = set(plays.loc[plays["desc"].eq("END GAME"), "game_id"])
pbp_game_ids = set(pbp_game_context["game_id"])
incomplete_game_ids = sorted(pbp_game_ids - completed_game_ids)
completion_summary = (
    game_context.assign(
        pbp_end_game=game_context["game_id"].isin(completed_game_ids),
        schedule_score_present=game_context[["home_score", "away_score"]].notna().all(axis=1),
    )
    .groupby(["season_pbp", "pbp_end_game", "schedule_score_present"], dropna=False)
    .size()
    .rename("games")
    .reset_index()
)
print(f"PBP games without END GAME marker (withheld): {incomplete_game_ids}")
completion_summary


PBP games without END GAME marker (withheld): []


,season_pbp,pbp_end_game,schedule_score_present,games
0,2023,True,True,285
1,2024,True,True,285
2,2025,True,True,285
3,2026,True,False,1
4,2026,True,True,79


## 4. Select standard pass/run plays and mark explosives

The source `success` flag is retained; its agreement with positive EPA is a validation, not a recalculation.


In [4]:
pass_run_plays = plays.loc[plays["play_type"].isin(["pass", "run"])].copy()
assert pass_run_plays["two_point_attempt"].notna().all()
assert pass_run_plays["two_point_attempt"].isin([0, 1]).all()

qualifying_plays = pass_run_plays.loc[
    pass_run_plays["game_id"].isin(completed_game_ids)
    & pass_run_plays["posteam"].notna()
    & pass_run_plays["defteam"].notna()
    & pass_run_plays["two_point_attempt"].eq(0)
].copy()
assert qualifying_plays[["epa", "success", "yards_gained"]].notna().all().all()
assert np.isfinite(qualifying_plays["epa"]).all()
assert qualifying_plays["success"].isin([0, 1]).all()
assert qualifying_plays["success"].eq(qualifying_plays["epa"].gt(0)).all()
assert qualifying_plays["posteam"].ne(qualifying_plays["defteam"]).all()

qualifying_plays["is_explosive"] = (
    (qualifying_plays["play_type"].eq("pass") & qualifying_plays["yards_gained"].ge(16))
    | (qualifying_plays["play_type"].eq("run") & qualifying_plays["yards_gained"].ge(12))
)
print(f"Qualifying plays: {len(qualifying_plays):,}")
qualifying_plays.groupby(["season", "season_type"]).agg(
    plays=("epa", "size"), explosive_plays=("is_explosive", "sum"),
)


Qualifying plays: 114,672


plays  explosive_plays
season season_type                        
2023   POST          1638              193
       REG          33836             3604
2024   POST          1567              191
       REG          33335             3747
2025   POST          1689              172
       REG          32813             3561
2026   REG           9794             1042

## 5. Aggregate offense and defense at team-game-week grain

The defense grouping uses the *same* play rows and signed EPA, but attributes them to `defteam`.


In [5]:
offense = (
    qualifying_plays
    .groupby(GROUP_KEYS + ["posteam", "defteam"], as_index=False)
    .agg(
        offensive_plays=("epa", "size"),
        offensive_epa_sum=("epa", "sum"),
        offensive_successful_plays=("success", "sum"),
        offensive_explosive_plays=("is_explosive", "sum"),
    )
    .rename(columns={"posteam": "team", "defteam": "opponent_team"})
)
defense = (
    qualifying_plays
    .groupby(GROUP_KEYS + ["defteam", "posteam"], as_index=False)
    .agg(
        defensive_plays_faced=("epa", "size"),
        defensive_epa_allowed_sum=("epa", "sum"),
    )
    .rename(columns={"defteam": "team", "posteam": "opponent_team"})
)
team_week_metrics = offense.merge(
    defense, on=GROUP_KEYS + ["team", "opponent_team"],
    how="outer", validate="one_to_one", indicator=True,
)
assert team_week_metrics["_merge"].eq("both").all()
team_week_metrics = team_week_metrics.drop(columns="_merge")

game_dates = schedules[["game_id", "gameday"]].rename(columns={"gameday": "game_date"})
team_week_metrics = team_week_metrics.merge(
    game_dates, on="game_id", how="left", validate="many_to_one",
)
assert team_week_metrics["game_date"].notna().all()
for count_column in [
    "offensive_plays", "offensive_successful_plays",
    "offensive_explosive_plays", "defensive_plays_faced",
]:
    team_week_metrics[count_column] = team_week_metrics[count_column].astype("int64")

team_week_metrics["offensive_epa_per_play"] = (
    team_week_metrics["offensive_epa_sum"] / team_week_metrics["offensive_plays"]
)
team_week_metrics["defensive_epa_per_play_allowed"] = (
    team_week_metrics["defensive_epa_allowed_sum"] / team_week_metrics["defensive_plays_faced"]
)
team_week_metrics["offensive_success_rate"] = (
    team_week_metrics["offensive_successful_plays"] / team_week_metrics["offensive_plays"]
)
team_week_metrics["offensive_explosive_play_rate"] = (
    team_week_metrics["offensive_explosive_plays"] / team_week_metrics["offensive_plays"]
)
team_week_metrics = team_week_metrics[OUTPUT_COLUMNS].sort_values(GOLD_KEY).reset_index(drop=True)
team_week_metrics.head()


,season,week,season_type,game_id,game_date,team,opponent_team,offensive_plays,offensive_epa_sum,offensive_successful_plays,offensive_explosive_plays,defensive_plays_faced,defensive_epa_allowed_sum,offensive_epa_per_play,defensive_epa_per_play_allowed,offensive_success_rate,offensive_explosive_play_rate
0,2023,1,REG,2023_01_ARI_WAS,2023-09-10,ARI,WAS,58,-20.418963,18,3,65,-19.121929,-0.352051,-0.294184,0.310345,0.051724
1,2023,1,REG,2023_01_ARI_WAS,2023-09-10,WAS,ARI,65,-19.121929,25,5,58,-20.418963,-0.294184,-0.352051,0.384615,0.076923
2,2023,1,REG,2023_01_BUF_NYJ,2023-09-11,BUF,NYJ,67,-4.541372,33,5,51,-2.418053,-0.067782,-0.047413,0.492537,0.074627
3,2023,1,REG,2023_01_BUF_NYJ,2023-09-11,NYJ,BUF,51,-2.418053,20,5,67,-4.541372,-0.047413,-0.067782,0.392157,0.098039
4,2023,1,REG,2023_01_CAR_ATL,2023-09-10,ATL,CAR,47,-5.046221,18,6,72,-19.426008,-0.107366,-0.269806,0.382979,0.127660


## 6. Validate game coverage, counts, rates, and signs

Every offensive play should appear exactly once as a defensive play faced by the opponent. Season summaries must use additive components, not mean weekly rates.


In [6]:
assert team_week_metrics.columns.tolist() == OUTPUT_COLUMNS
assert team_week_metrics[GOLD_KEY].notna().all().all()
assert not team_week_metrics.duplicated(GOLD_KEY).any()
assert team_week_metrics.groupby("game_id").size().eq(2).all()
assert set(team_week_metrics["game_id"]) == completed_game_ids
assert team_week_metrics[["offensive_plays", "defensive_plays_faced"]].gt(0).all().all()
assert team_week_metrics["offensive_successful_plays"].between(0, team_week_metrics["offensive_plays"]).all()
assert team_week_metrics["offensive_explosive_plays"].between(0, team_week_metrics["offensive_plays"]).all()
for rate_column in ["offensive_success_rate", "offensive_explosive_play_rate"]:
    assert team_week_metrics[rate_column].between(0, 1).all()
assert np.isfinite(team_week_metrics[["offensive_epa_per_play", "defensive_epa_per_play_allowed"]]).all().all()

team_schedule_check = team_week_metrics.merge(
    schedules[["game_id", "home_team", "away_team"]],
    on="game_id", how="left", validate="many_to_one",
)
assert (
    team_schedule_check["team"].eq(team_schedule_check["home_team"])
    & team_schedule_check["opponent_team"].eq(team_schedule_check["away_team"])
    | team_schedule_check["team"].eq(team_schedule_check["away_team"])
    & team_schedule_check["opponent_team"].eq(team_schedule_check["home_team"])
).all()

opponent_defense = team_week_metrics[[
    "game_id", "team", "defensive_plays_faced", "defensive_epa_allowed_sum",
]].rename(columns={
    "team": "opponent_team",
    "defensive_plays_faced": "opponent_defensive_plays_faced",
    "defensive_epa_allowed_sum": "opponent_defensive_epa_allowed_sum",
})
mirror_check = team_week_metrics.merge(
    opponent_defense, on=["game_id", "opponent_team"],
    how="left", validate="many_to_one",
)
assert mirror_check["offensive_plays"].eq(mirror_check["opponent_defensive_plays_faced"]).all()
assert np.allclose(mirror_check["offensive_epa_sum"], mirror_check["opponent_defensive_epa_allowed_sum"])
assert team_week_metrics["offensive_plays"].sum() == len(qualifying_plays)
assert team_week_metrics["offensive_successful_plays"].sum() == qualifying_plays["success"].sum()
assert team_week_metrics["offensive_explosive_plays"].sum() == qualifying_plays["is_explosive"].sum()
assert np.isclose(team_week_metrics["offensive_epa_sum"].sum(), qualifying_plays["epa"].sum())

for numerator, denominator, rate in [
    ("offensive_epa_sum", "offensive_plays", "offensive_epa_per_play"),
    ("defensive_epa_allowed_sum", "defensive_plays_faced", "defensive_epa_per_play_allowed"),
    ("offensive_successful_plays", "offensive_plays", "offensive_success_rate"),
    ("offensive_explosive_plays", "offensive_plays", "offensive_explosive_play_rate"),
]:
    assert np.allclose(team_week_metrics[rate], team_week_metrics[numerator] / team_week_metrics[denominator])

validation_summary = (
    team_week_metrics.groupby(["season", "season_type"], as_index=False)
    .agg(games=("game_id", "nunique"), team_rows=("team", "size"),
         offensive_plays=("offensive_plays", "sum"),
         defensive_plays_faced=("defensive_plays_faced", "sum"))
)
assert validation_summary["team_rows"].eq(2 * validation_summary["games"]).all()
assert validation_summary["offensive_plays"].eq(validation_summary["defensive_plays_faced"]).all()
validation_summary


,season,season_type,games,team_rows,offensive_plays,defensive_plays_faced
0,2023,POST,13,26,1638,1638
1,2023,REG,272,544,33836,33836
2,2024,POST,13,26,1567,1567
3,2024,REG,272,544,33335,33335
4,2025,POST,13,26,1689,1689
5,2025,REG,272,544,32813,32813
6,2026,REG,80,160,9794,9794


## 7. Persist and round-trip each season


In [7]:
GOLD_DIR.mkdir(parents=True, exist_ok=True)
output_summary = []
for season in SEASONS:
    season_metrics = team_week_metrics.loc[team_week_metrics["season"].eq(season)].reset_index(drop=True)
    assert not season_metrics.empty
    output_path = GOLD_DIR / f"gold_team_week_metrics_{season}.parquet"
    season_metrics.to_parquet(output_path, index=False)
    reloaded = pd.read_parquet(output_path)
    pd.testing.assert_frame_equal(reloaded, season_metrics)
    assert reloaded.columns.tolist() == OUTPUT_COLUMNS
    assert reloaded["season"].eq(season).all()
    output_summary.append({
        "season": season, "games": reloaded["game_id"].nunique(),
        "team_rows": len(reloaded), "columns": len(reloaded.columns),
        "output": str(output_path.relative_to(PROJECT_ROOT)),
    })
    print(f"Validated and wrote {output_path.name}")
pd.DataFrame(output_summary)


Validated and wrote gold_team_week_metrics_2023.parquet
Validated and wrote gold_team_week_metrics_2024.parquet
Validated and wrote gold_team_week_metrics_2025.parquet
Validated and wrote gold_team_week_metrics_2026.parquet


,season,games,team_rows,columns,output
0,2023,285,570,17,data/gold/team_week_metrics/gold_team_week_met...
1,2024,285,570,17,data/gold/team_week_metrics/gold_team_week_met...
2,2025,285,570,17,data/gold/team_week_metrics/gold_team_week_met...
3,2026,80,160,17,data/gold/team_week_metrics/gold_team_week_met...
